In [5]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Arduino Data Logger Analysis\n",
    "\n",
    "This notebook loads and visualizes the serial data logged from the Arduino robot.\n",
    "\n",
    "Assumptions:\n",
    "- The data file is named `data.txt` and is located in the same directory as this notebook.\n",
    "- Data is space-separated with the following columns (based on the Serial.print statements):\n",
    "  1. CompInput state (int)\n",
    "  2. analogRead(0) (int)\n",
    "  3. CompInput state (string: '0' or '1')\n",
    "  4. Robot state (string: '1', '2', or '3')\n",
    "  5. timeNow (unsigned long)\n",
    "  6. timeDelta (unsigned int)\n",
    "  7. extensionFireDelay (int)\n",
    "  8. extensionRetractDelay (int)\n",
    "  9. driveTime (long int)\n",
    "  10. marioLift.getPosition() (long int)\n",
    "  11. liftSwitch.getValInt() (int)\n",
    "  12. marioLift.atBottomLimit() (bool, as int)\n",
    "  13. lumaTriggered (bool, as int)\n",
    "  14. currentLumaFireDelay (long int)\n",
    "  15. currentLumaRetractDelay (long int)\n",
    "  16. koopaDelay (long int)\n",
    "\n",
    "Adjust the file path and column names as needed."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 1,
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "import numpy as np\n",
    "from IPython.display import display\n",
    "\n",
    "# Load the data with error handling\n",
    "file_path = 'data/FILE00053.txt'\n",
    "try:\n",
    "    df = pd.read_csv(file_path, sep=' ', header=None, names=[\n",
    "        'CompInput_state_int', 'analogRead_0', 'CompInput_state_str', 'robot_state_str',\n",
    "        'timeNow', 'timeDelta', 'extensionFireDelay', 'extensionRetractDelay',\n",
    "        'driveTime', 'liftPosition', 'liftSwitch', 'atBottomLimit',\n",
    "        'lumaTriggered', 'currentLumaFireDelay', 'currentLumaRetractDelay', 'koopaDelay'\n",
    "    ], on_bad_lines='skip', engine='python')\n",
    "    \n",
    "    # Remove rows with NaN values\n",
    "    df = df.dropna()\n",
    "    \n",
    "    # Convert timeNow to seconds for plotting\n",
    "    df['time_seconds'] = df['timeNow'] / 1000.0\n",
    "    \n",
    "    print(f\"Loaded {len(df)} rows of data\")\n",
    "    print(f\"Data shape: {df.shape}\")\n",
    "    display(df.head())\n",
    "    \n",
    "except Exception as e:\n",
    "    print(f\"Error loading file: {e}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Plot Key Variables Over Time\n",
    "\n",
    "Select and plot variables of interest. For example, lift position, drive time, and robot state."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 1,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Plot lift position over time\n",
    "plt.figure(figsize=(12, 6))\n",
    "plt.subplot(2, 2, 1)\n",
    "plt.plot(df['time_seconds'], df['liftPosition'], label='Lift Position')\n",
    "plt.xlabel('Time (s)')\n",
    "plt.ylabel('Lift Position')\n",
    "plt.title('Lift Position Over Time')\n",
    "plt.legend()\n",
    "\n",
    "# Plot drive time over time\n",
    "plt.subplot(2, 2, 2)\n",
    "plt.plot(df['time_seconds'], df['driveTime'], label='Drive Time', color='orange')\n",
    "plt.xlabel('Time (s)')\n",
    "plt.ylabel('Drive Time')\n",
    "plt.title('Drive Time Over Time')\n",
    "plt.legend()\n",
    "\n",
    "# Plot robot state (as numeric for simplicity)\n",
    "plt.subplot(2, 2, 3)\n",
    "plt.plot(df['time_seconds'], df['robot_state_str'].astype(int), label='Robot State', color='green')\n",
    "plt.xlabel('Time (s)')\n",
    "plt.ylabel('Robot State (1=Waiting, 2=Moving, 3=Center)')\n",
    "plt.title('Robot State Over Time')\n",
    "plt.legend()\n",
    "\n",
    "# Plot koopa delay over time\n",
    "plt.subplot(2, 2, 4)\n",
    "plt.plot(df['time_seconds'], df['koopaDelay'], label='Koopa Delay', color='red')\n",
    "plt.xlabel('Time (s)')\n",
    "plt.ylabel('Koopa Delay')\n",
    "plt.title('Koopa Delay Over Time')\n",
    "plt.legend()\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Interactive Exploration\n",
    "\n",
    "Use the widgets below to select and plot different variables interactively."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 1,
   "metadata": {},
   "outputs": [],
   "source": [
    "from ipywidgets import interact, Dropdown\n",
    "\n",
    "# Function to plot selected variable\n",
    "def plot_variable(var_name):\n",
    "    plt.figure(figsize=(10, 5))\n",
    "    plt.plot(df['time_seconds'], df[var_name])\n",
    "    plt.xlabel('Time (s)')\n",
    "    plt.ylabel(var_name)\n",
    "    plt.title(f'{var_name} Over Time')\n",
    "    plt.show()\n",
    "\n",
    "# Dropdown for variable selection\n",
    "var_dropdown = Dropdown(options=df.columns.tolist(), value='liftPosition', description='Variable:')\n",
    "interact(plot_variable, var_name=var_dropdown)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Summary Statistics\n",
    "\n",
    "Get quick stats on the data."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 1,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Summary statistics\n",
    "df.describe()"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.8.5"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}

{'cells': [{'cell_type': 'markdown',
   'metadata': {},
   'source': ['# Arduino Data Logger Analysis\n',
    '\n',
    'This notebook loads and visualizes the serial data logged from the Arduino robot.\n',
    '\n',
    'Assumptions:\n',
    '- The data file is named `data.txt` and is located in the same directory as this notebook.\n',
    '- Data is space-separated with the following columns (based on the Serial.print statements):\n',
    '  1. CompInput state (int)\n',
    '  2. analogRead(0) (int)\n',
    "  3. CompInput state (string: '0' or '1')\n",
    "  4. Robot state (string: '1', '2', or '3')\n",
    '  5. timeNow (unsigned long)\n',
    '  6. timeDelta (unsigned int)\n',
    '  7. extensionFireDelay (int)\n',
    '  8. extensionRetractDelay (int)\n',
    '  9. driveTime (long int)\n',
    '  10. marioLift.getPosition() (long int)\n',
    '  11. liftSwitch.getValInt() (int)\n',
    '  12. marioLift.atBottomLimit() (bool, as int)\n',
    '  13. lumaTriggered (bool, as int)\n'

In [14]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

# Load the data with error handling
file_path = 'data/FILE00053.txt'
try:
    df = pd.read_csv(file_path, sep=' ', header=None, names=[
        'CompInput_state_int', 'analogRead_0', 'CompInput_state_str', 'robot_state_str',
        'timeNow', 'timeDelta', 'extensionFireDelay', 'extensionRetractDelay',
        'driveTime', 'liftPosition', 'liftSwitch', 'atBottomLimit',
        'lumaTriggered', 'currentLumaFireDelay', 'currentLumaRetractDelay', 'koopaDelay'
    ], on_bad_lines='skip', engine='python')
    
    # Remove rows with NaN values
    df = df.dropna()
    
    # Convert string columns to int for plotting
    df['CompInput_state_str'] = df['CompInput_state_str'].astype(int)
    df['robot_state_str'] = df['robot_state_str'].astype(int)
    df['analogRead_0'] = df['analogRead_0'].astype(int)
    
    # Convert timeNow to seconds for plotting
    df['time_seconds'] = df['timeNow'] / 1000.0
    
    print(f"Loaded {len(df)} rows of data")
    print(f"Data shape: {df.shape}")
    display(df.head())
    
except Exception as e:
    print(f"Error loading file: {e}")

Loaded 5720 rows of data
Data shape: (5720, 17)


,CompInput_state_int,analogRead_0,CompInput_state_str,robot_state_str,timeNow,timeDelta,extensionFireDelay,extensionRetractDelay,driveTime,liftPosition,liftSwitch,atBottomLimit,lumaTriggered,currentLumaFireDelay,currentLumaRetractDelay,koopaDelay,time_seconds
1,0,1012,0,1,0.0,0.0,0.0,1000.0,7000.0,0.0,1.0,1.0,0.0,1000.0,50.0,35000.0,0.000
2,0,1019,0,1,22.0,22.0,0.0,1000.0,7000.0,0.0,1.0,1.0,0.0,1000.0,50.0,35000.0,0.022
3,0,1014,0,1,45.0,23.0,0.0,1000.0,7000.0,0.0,1.0,1.0,0.0,1000.0,50.0,35000.0,0.045
4,0,1017,0,1,67.0,22.0,0.0,1000.0,7000.0,0.0,1.0,1.0,0.0,1000.0,50.0,35000.0,0.067
5,0,1023,0,1,90.0,23.0,0.0,1000.0,7000.0,0.0,1.0,1.0,0.0,1000.0,50.0,35000.0,0.090


In [13]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Create interactive subplots
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Raw Competition Input', 'Drive Time', 'Robot State', 'Koopa Delay')
)

# Raw Competition Input
fig.add_trace(
    go.Scatter(x=df['time_seconds'], y=df['analogRead_0'], mode='lines', name='Raw Competition Input'),
    row=1, col=1
)

# Drive Time
fig.add_trace(
    go.Scatter(x=df['time_seconds'], y=df['driveTime'], mode='lines', name='Drive Time', line=dict(color='orange')),
    row=1, col=2
)

# Robot State
fig.add_trace(
    go.Scatter(x=df['time_seconds'], y=df['robot_state_str'].astype(int), mode='lines', name='Robot State', line=dict(color='green')),
    row=2, col=1
)

# Koopa Delay
fig.add_trace(
    go.Scatter(x=df['time_seconds'], y=df['koopaDelay'], mode='lines', name='Koopa Delay', line=dict(color='red')),
    row=2, col=2
)

# Update layout
fig.update_xaxes(title_text='Time (s)', row=1, col=1)
fig.update_xaxes(title_text='Time (s)', row=1, col=2)
fig.update_xaxes(title_text='Time (s)', row=2, col=1)
fig.update_xaxes(title_text='Time (s)', row=2, col=2)

fig.update_yaxes(title_text='Position', row=1, col=1)
fig.update_yaxes(title_text='Drive Time', row=1, col=2)
fig.update_yaxes(title_text='State', row=2, col=1)
fig.update_yaxes(title_text='Delay (ms)', row=2, col=2)

fig.update_layout(height=800, showlegend=True, title_text='Stormkoopas Data Analysis')
fig.show()


In [15]:
import plotly.express as px

# Create dropdown for variable selection
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
numeric_cols.remove('time_seconds')  # Remove time column from dropdown

# Create figure with all traces initially hidden
fig = go.Figure()

for col in numeric_cols:
    fig.add_trace(
        go.Scatter(x=df['time_seconds'], y=df[col], mode='lines', name=col, visible=False)
    )

fig.data[0].visible = True

# Create buttons for dropdown
buttons = [dict(label=col, method='update', 
                args=[{'visible': [col == trace.name for trace in fig.data]},
                      {'title': f'{col} Over Time', 'yaxis': {'title': col}}])
           for col in numeric_cols]

fig.update_layout(
    updatemenus=[dict(active=0, buttons=buttons)],
    title=numeric_cols[0] + ' Over Time',
    xaxis_title='Time (s)',
    yaxis_title=numeric_cols[0],
    hovermode='x unified',
    height=600
)

fig.show()